# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"
# create llm assistant
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-11-24 02:10:35.74][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


## Extract Features **X** Used in Model

In [3]:
# create dict to store features
# features = {}

In [4]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # create internal dict for analysis features
#     features[i] = {}
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
#     control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in ind_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     for dict_idx, var in enumerate(ind_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
        
#     # save updated independent variables in features dict
#     features[i]['independent_variables'] = ind_vars
    
#     # tkae same approach for control variables
#     for dict_idx, var in enumerate(control_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
    
#     # save updated control variables in features dict
#     features[i]['control_variables'] = control_vars

In [5]:
# view feature dictionary to ensure correctness
# features

## Extract Response *y* used in Model

In [6]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in response_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     # for dict_idx, var in enumerate(response_vars):
#     transform_responses = get_feature_transforms(llm_assistant,
#                                                  transform_code,
#                                                  response_vars['columns'],
#                                                  response_vars['description'])
#     response_vars['transform_code'] = [response.text[0].content \
#         for response in transform_responses]

#     # save updated response variables in features dict
#     features[i]['response_variables'] = response_vars

In [7]:
# view feature dictionary to ensure correctness
# features

## Extract Features **X** and *y* Used in Model

In [8]:
features = format_features(multirun_analyses, num_analyses, llm_assistant)

[2025-11-24 02:10:36.77][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:10:53.43][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  16.66 seconds
[2025-11-24 02:10:53.44][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:10:53.46][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:11:06.04][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.58 seconds
[2025-11-24 02:11:06.05][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:11:06.08][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:11:15.05][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  8.9

In [9]:
features

{0: {'independent_variables': [{'description': 'Continuous masculinity-femininity rating of the hurricane name (higher = more feminine). This is the primary independent variable capturing perceived femininity of the name, standardized (z-scored) for modelling.',
    'columns': ['masfem_z'],
    'transform_code': ["for col in ['masfem', 'gender_mf', 'alldeaths', 'wind', 'category', 'min', 'year']:\n    if col in df.columns:\n        df[col] = pd.to_numeric(df[col], errors='coerce')\n\ndf['masfem_z'] = (df['masfem'] - df['masfem'].mean()) / (df['masfem'].std(ddof=0) if df['masfem'].std(ddof=0) != 0 else 1.0)"]},
   {'description': 'Binary indicator of hurricane name gender (0 = male, 1 = female). Included as an alternative operationalization of name gender.',
    'columns': ['gender_mf'],
    'transform_code': ["for col in ['masfem', 'gender_mf', 'alldeaths', 'wind', 'category', 'min', 'year']:\n    if col in df.columns:\n        df[col] = pd.to_numeric(df[col], errors='coerce')\n\ndf['g

## Extract Model Class Used

In [10]:
model_info = format_model_info(multirun_analyses, num_analyses, llm_assistant)
model_info

[2025-11-24 02:14:08.21][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:14:25.30][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  17.09 seconds
[2025-11-24 02:14:25.31][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:14:25.33][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:14:35.64][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  10.31 seconds
[2025-11-24 02:14:35.65][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:14:35.67][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:14:49.03][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  13.

{0: '{\n  "model_library": "statsmodels",\n  "model_class": "sm.GLM (NegativeBinomial, with Poisson fallback) and sm.OLS",\n  "model_parameters": "GLM: family=sm.families.NegativeBinomial() (fallback: family=sm.families.Poisson()); no explicit link or dispersion parameter provided; default fit() options used. OLS: sm.OLS(...).fit() with default options. A constant is added via sm.add_constant(X). No regularization, weights, or robust covariance options are explicitly set in the code.",\n  "model_formula_fitting_code": "X_cols = [c for c in [\'masfem_z\', \'StormSeverity\', \'year_cent\', \'gender_mf\'] if c in df.columns]\\nX = df[X_cols].astype(float)\\nX = sm.add_constant(X)\\n\\ny_count = df[\'alldeaths\'].astype(float)\\n\\n# Fit Negative Binomial GLM (fallback to Poisson on exception)\\ntry:\\n    nb_model = sm.GLM(y_count, X, family=sm.families.NegativeBinomial()).fit()\\nexcept Exception:\\n    poisson = sm.GLM(y_count, X, family=sm.families.Poisson()).fit()\\n    nb_model = poi

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [11]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [12]:
# Run transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    try:
        transformed_datasets[i] = transform_func(data.copy())  # use copy of dataset
        print(f"[Transform {i}] Completed successfully.")
    except Exception as e:
        print(f"[Transform {i}] Failed with error: {e}")
        transformed_datasets[i] = None

# Run model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    try:
        if transformed_datasets[i] is None:
            print(f"[Model {i}] Skipping — transform step failed.")
            continue

        model_results[i] = model_func(transformed_datasets[i].copy())  # use copy
        print(f"[Model {i}] Completed successfully.")
    except Exception as e:
        print(f"[Model {i}] Failed with error: {e}")
        model_results[i] = None


[Transform 0] Completed successfully.
[Transform 1] Completed successfully.
[Transform 2] Completed successfully.


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[Model 0] Completed successfully.
MAIN MODEL: LogDamage ~ Fem_Z + controls
                            OLS Regression Results                            
Dep. Variable:              LogDamage   R-squared:                       0.550
Model:                            OLS   Adj. R-squared:                  0.507
Method:                 Least Squares   F-statistic:                     2048.
Date:                Mon, 24 Nov 2025   Prob (F-statistic):           1.88e-94
Time:                        14:14:57   Log-Likelihood:                -173.87
No. Observations:                  93   AIC:                             365.7
Df Residuals:                      84   BIC:                             388.5
Df Model:                           8                                         
Covariance Type:                  HC3                                         
                                                                       coef    std err          z      P>|z|      [0.025      0.975]
--

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [13]:
# view the first model result as a sanity check
model_results[0]

{'nb_model': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x79e4cff958a0>,
 'ols_model': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x79e4cff96710>,
 'secondary_models': {'nb_binary_gender': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x79e4cff97910>,
  'ols_binary_gender': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x79e4cff97940>},
 'specification': {'X_columns': ['masfem_z',
   'StormSeverity',
   'year_cent',
   'gender_mf'],
  'outcome_count': 'alldeaths',
  'outcome_log': 'log_alldeaths'}}

In [14]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-11-24 02:14:57.50][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:15:28.50][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  31.00 seconds
[2025-11-24 02:15:28.50][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:15:28.52][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:15:49.73][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  21.22 seconds
[2025-11-24 02:15:49.74][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:15:49.77][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:16:20.09][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  30.

In [15]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts statistics for the effect of hurricane name femininity on deaths from the provided\n    model_output (as returned by the modeling function). Returns a dictionary with keys:\n      - "object": dict of extracted numeric results for each relevant model and a short conclusion\n      - "description": human-readable explanation of what the numbers mean for the hypothesis\n\n    The function looks for:\n      - \'masfem_z\' coefficient in the primary negative-binomial (or Poisson fallback) and OLS models\n      - \'gender_mf\' coefficient in the secondary binary-gender models (if present)\n\n    Interpretation rule used for the conclusion:\n      - The hypothesis (more feminine names -> fewer precautions -> more deaths) is supported\n        if we observe a positive coefficient for the femininity variable and p-value < 0.05\n        in at least one primary model. If a strong negative significant coefficient appears,\n        

In [16]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [17]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    try:
        interpretation_code = final_answer_code[i]
    except (KeyError, IndexError, TypeError):
        interpretation_code = None
    
    try: 
        interpretation_output = final_answers[i]
    except (KeyError, IndexError, TypeError):
        interpretation_output = None
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-11-24 02:16:20.49][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:16:27.95][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.46 seconds
[2025-11-24 02:16:27.96][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:16:27.98][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:16:32.52][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  4.54 seconds
[2025-11-24 02:16:32.53][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-24 02:16:32.55][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:16:38.51][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  5.96 

In [18]:
conclusions

{0: '{\n  "answer": "Yes",\n  "justification": "The negative-binomial count models show a positive, statistically significant association between name femininity and deaths (masfem_z coef = 0.973, p = 0.007 → ≈165% higher expected deaths per SD increase; binary female coef = 1.294, p < 1e-7 → ≈265% higher). The OLS on log deaths is not significant, but the primary count-model evidence supports the hypothesis."\n}',
 1: '{\n  "answer": "No",\n  "justification": "The main model\'s coefficient on femininity (Fem_Z) is positive (0.0847) but not statistically significant (p = 0.6269; 95% CI includes zero). Robustness checks (deaths outcome and MTurk-rated femininity) likewise show non-significant effects. Thus the evidence does not support the hypothesis."\n}',
 2: '{\n  "answer": "No",\n  "justification": "The primary OLS on log fatalities shows a non‑significant coefficient (coef = -0.253, SE = 0.361, p = 0.484). The Negative Binomial robustness also yields a non‑significant effect (IRR ≈

In [19]:
llm_judge = llm(provider=llm_provider, model=llm_model)

[2025-11-24 02:16:38.81][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [20]:
data_head = data.head(10)

In [21]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final JSON object.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in JSON format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)

In [22]:
judge_user_prompt = (
    f"Research Question / Context:\n{task}\n\n"
    "Here is a sample of the dataset to understand the structure and variables:\n"
    f"{data_head}\n\n"
    "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
    "==================== TRIAL 0 ====================\n\n"
    "Independent Variables:\n"
    f"{features[0]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[0]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[0]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[0]}\n\n"
    "Conclusion:\n"
    f"{conclusions[0]}\n\n"
    "==================== TRIAL 1 ====================\n\n"
    "Independent Variables:\n"
    f"{features[1]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[1]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[1]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[1]}\n\n"
    "Conclusion:\n"
    f"{conclusions[1]}\n\n"
    "Now, following your reasoning plan, provide similarity ratings as JSON only."
)

In [23]:
final_scores = llm_judge.generate([{"role": "system",
                                        "content": judge_system_prompt},
                                       {"role": "user",
                                        "content": judge_user_prompt}])

[2025-11-24 02:16:39.39][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-24 02:16:46.80][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  7.41 seconds
[2025-11-24 02:16:46.80][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
